In [52]:
import os
from glob import glob
from subprocess import run
from io import StringIO

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

os.chdir('/home/vladimirnoz/Projects/Codebook_Perspectives/')

In [7]:
# number of matrices

len(glob('motifs/manual_motifs_v2/pwms/*.pwm'))

235

In [8]:
# number of peaks

len(glob('peaks/chipseq+ghtselex/*.bed'))

216

In [9]:
# number of TOPs

len(glob('TOPs/TOPs/*.bed'))

137

# ASB Stats

In [26]:
# chip-seq

#num_snps
snps = glob('AS_CHS_GHTS/SNPs_filtered/chipseq/*/*.vcf.gz')
total_snps = 0
for path in tqdm(snps):
    cmd = f'bcftools view -H {path} | wc -l'
    output = run(cmd, shell=True, text=True, capture_output=True).stdout
    num = int(output.strip())
    total_snps += num
print(total_snps)

  0%|          | 0/361 [00:00<?, ?it/s]

889814


In [28]:
# chip-seq

#num_tfs
tfs = glob('AS_CHS_GHTS/SNPs_filtered/chipseq/*')
len(tfs)

155

In [61]:
# chip-seq stats

datasets = glob('AS_CHS_GHTS/as_tables/chipseq/ASB_motifs/*.tsv')
total_asb = 0
pwm_hits = 0
concordant_hits = 0
strongly_concordant = 0

for path in tqdm(datasets):
    df = pd.read_table(path).query('fdr_comb_pval < 0.05')
    if len(df) == 0:
        continue
    total_asb += len(df)

    pwm_hits += len(df.query('motif_conc != "No Hit"'))
    concordant_hits += len(df.query('motif_conc == "Concordant"'))
    strongly_concordant += len(df.query('motif_conc == "Concordant" & abs(motif_fc) >= 1'))

    

print(total_asb)
print(pwm_hits, f'{pwm_hits/total_asb:.0%}')
print(concordant_hits, f'{concordant_hits/pwm_hits:.0%}')
print(strongly_concordant)

  0%|          | 0/155 [00:00<?, ?it/s]

10575
2809 27%
1897 68%
1373


In [29]:
# ght-selex

#num_snps
snps = glob('AS_CHS_GHTS/SNPs_filtered/selex/*/*.vcf.gz')
total_snps = 0
for path in tqdm(snps):
    cmd = f'bcftools view -H {path} | wc -l'
    output = run(cmd, shell=True, text=True, capture_output=True).stdout
    num = int(output.strip())
    total_snps += num
print(total_snps)

  0%|          | 0/1303 [00:00<?, ?it/s]

35183


In [30]:
# ght-selex

#num_tfs
tfs = glob('AS_CHS_GHTS/SNPs_filtered/selex/*')
len(tfs)

178

In [34]:
# ght-selex

#num_experiments
datasets = glob('AS_CHS_GHTS/SNPs_filtered/selex/*/*')
len(set([x.split('.')[1] for x in datasets]))

370

In [62]:
# ght-selex stats

datasets = glob('AS_CHS_GHTS/as_tables/selex/ASB_motifs/*.tsv')
total_asb = 0
pwm_hits = 0
concordant_hits = 0
strongly_concordant = 0

for path in tqdm(datasets):
    df = pd.read_table(path).query('fdr_comb_pval < 0.05')
    if len(df) == 0:
        continue
    total_asb += len(df)

    pwm_hits += len(df.query('motif_conc != "No Hit"'))
    concordant_hits += len(df.query('motif_conc == "Concordant"'))
    strongly_concordant += len(df.query('motif_conc == "Concordant" & abs(motif_fc) >= 1'))

    

print(total_asb)
print(pwm_hits, f'{pwm_hits/total_asb:.0%}')
print(concordant_hits, f'{concordant_hits/pwm_hits:.0%}')
print(strongly_concordant)

  0%|          | 0/178 [00:00<?, ?it/s]

1485
755 51%
469 62%
309


In [46]:
# Total ASBs

datasets = glob('AS_CHS_GHTS/as_tables/*/ASB_motifs/*.tsv')
total_asb = 0
pwm_hits = 0
concordant_hits = 0

for path in tqdm(datasets):
    df = pd.read_table(path).query('fdr_comb_pval < 0.05')
    if len(df) == 0:
        continue
    total_asb += len(df)

    pwm_hits += len(df.query('motif_conc != "No Hit"'))
    concordant_hits += len(df.query('motif_conc == "Concordant"'))

    

print(total_asb)
print(pwm_hits, f'{pwm_hits/total_asb:.0%}')
print(concordant_hits, f'{concordant_hits/pwm_hits:.0%}')

  0%|          | 0/333 [00:00<?, ?it/s]

12060
3564 30%
2366 66%


In [59]:
# ASBs in TOPs

datasets = glob('AS_CHS_GHTS/as_tables/*/ASB_motifs/*.tsv')
asbs_in_tops = 0


for path in tqdm(datasets):
    df = pd.read_table(path).query('fdr_comb_pval < 0.05')
    tf = os.path.basename(path).replace('.tsv', '')
    top_path = f'TOPs/TOPs/{tf}.bed'
    cmd = f'bedtools intersect -a stdin -b {top_path}'
    df_text = df.to_csv(index=False, sep='\t')
    proc = run(cmd, shell=True, text=True, capture_output=True, input=df_text)
    output = proc.stdout
    asbs_in_tops += len(output.split('\n'))
asbs_in_tops

  0%|          | 0/333 [00:00<?, ?it/s]

909

# Chromatin

In [65]:
# total datasets

!ls Chromatin/as_tables/ | wc -l

211030


In [84]:
# motif-annotated sets of ASchromatin sites

len(pd.read_table('Chromatin/agg_dnase/pvalues_init.tsv')) + len(pd.read_table('Chromatin/agg_atac/pvalues_init.tsv'))

7070

In [89]:
len(pd.read_table('Chromatin/agg_atac/pvalues_agg_greater.tsv'))

219

In [88]:
len(pd.read_table('Chromatin/agg_atac/pvalues_agg_greater.tsv').query('fdr_comb < 0.05'))

20

In [90]:
len(pd.read_table('Chromatin/agg_dnase/pvalues_agg_greater.tsv'))

215

In [87]:
len(pd.read_table('Chromatin/agg_dnase/pvalues_agg_greater.tsv').query('fdr_comb < 0.05'))

48